# Fit a compact model and inspect the complete study

This notebook performs a small real normal-only teaching fit, then inspects the separate completed source-grouped evaluation. Its didactic training error is not a test result.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p/'worldmodels/v2_data.py').is_file()), None)
if PROJECT is None:
    PROJECT = next(p/'experiments/aaai-worldmodels' for p in candidates if (p/'experiments/aaai-worldmodels/worldmodels/v2_data.py').is_file())
sys.path.insert(0, str(PROJECT))
REPOSITORY = PROJECT.parents[1]
from worldmodels.v2_data import load_cohort
from worldmodels import v2_evaluation as ev
from worldmodels import v2_models as models
from worldmodels.v2_runner import window_errors, verify_freeze
COHORT = PROJECT/'artifacts/runs/reference-v2-cohort-v1'
RUN = PROJECT/'artifacts/runs/reference-v2-core-v1'
TRANSFER = PROJECT/'artifacts/runs/reference-v2-transfer-v1'
FIGURES = PROJECT/'artifacts/executed-reference-v2/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
USED_INPUTS={}
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def record_input(path):
    path=Path(path).resolve()
    key=str(path.relative_to(REPOSITORY)) if path.is_relative_to(REPOSITORY) else str(path)
    USED_INPUTS[key]=sha(path)
def read_json(path):
    path = Path(path)
    assert path.is_file(), f'Required completed artifact is absent: {path}. Read docs/reference_v2/README.md for prerequisites.'
    record_input(path)
    return json.loads(path.read_text())
def read_csv(path, **kwargs):
    record_input(path)
    return pd.read_csv(path, **kwargs)
def artifact(name, directory=RUN):
    path = directory/name
    manifest = read_json(directory/'artifact_hashes.json')
    assert name in manifest and path.is_file(), f'Unrecorded or missing artifact: {path}'
    assert sha(path) == manifest[name], f'Artifact hash differs from the completed run: {path}'
    record_input(path)
    return path
def completed_run():
    result = read_json(artifact('results.json'))
    assert result['status']=='complete' and not result.get('test_only', False)
    return result
def save_figure(figure, name):
    figure.savefig(FIGURES/(name+'.svg'), bbox_inches='tight')
    figure.savefig(FIGURES/(name+'.png'), dpi=150, bbox_inches='tight')
freeze = verify_freeze()
record_input(PROJECT/'docs/reference_v2/protocol_freeze.json')
x, valid, metadata, actual_times = load_cohort(COHORT, verify=True)
cohort_identity=read_json(COHORT/'identity.json');record_input(COHORT/'cohort.csv')
for name in cohort_identity['prepared_inputs']:
    record_input(REPOSITORY/cohort_identity['prepared_directory']/name)
assert x.shape == (len(metadata),24,16,2) and len(metadata)==2815
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'svg.fonttype':'none','axes.spines.top':False,'axes.spines.right':False})
print('Verified real observations:',len(metadata),'windows;',metadata.source_id.nunique(),'source videos.')
print('Observation units: pelvis-relative 2D coordinates divided by a prefix torso scale. Times are decoder seconds.')


## Fit a small real normal-only teaching model

This cell performs a real fit on a declared subset of normal sources outside fold zero. It uses a fixed penalty and a few windows per source for a quick demonstration. The displayed training error is not held-out research evidence and does not retune the completed study.

In [ ]:
from worldmodels.v2_models import ForecastConfig,fit_forecaster,predict_forecaster
training=metadata[metadata.label.eq('normal') & metadata.outer_fold.ne(0)].groupby('source_id',sort=True).head(4).index.to_numpy()
training_meta=metadata.iloc[training].copy();training_meta['group_id']=training_meta.component_id
configuration=ForecastConfig(mode='ridge_posture_velocity',ridge_alpha=10.,seed=27)
teaching_model=fit_forecaster(x[training,:16],valid[training,:16],actual_times[training,:16],x[training,16:],valid[training,16:],training_meta,configuration)
training_prediction=predict_forecaster(teaching_model,x[training,:16],valid[training,:16],actual_times[training,:16])
teaching_error=ev.aggregate_windows(training_meta,window_errors(x[training],valid[training],training_prediction))
print('Actual fit:',len(training),'windows;',training_meta.source_id.nunique(),'normal source groups;',round(teaching_model['seconds'],3),'seconds.')
print('Didactic training MSE, not evaluation MSE:',float(teaching_error.value.mean()))


## Inspect how the compact state represents movement

The posture model receives the latest distal-joint positions. The posture-plus-velocity model adds recent median coordinate slopes, their dispersion, and observation counts. A small residual neural model receives the full observed prefix. These feature sets ask different questions about whether history adds useful information.

In [ ]:
dimensions=[]
for mode in ['ridge_posture','ridge_posture_velocity','mlp']:
    features,anchors,known=models.causal_features(x[training[:3],:16],valid[training[:3],:16],actual_times[training[:3],:16],mode=mode)
    dimensions.append({'mode':mode,'input_features':features.shape[1],'future_coordinate_queries':8*8*2})
display(pd.DataFrame(dimensions))
weights=models.hierarchical_weights(training_meta)
mass=training_meta.assign(weight=weights).groupby('source_id').weight.sum()
np.testing.assert_allclose(mass.to_numpy(),mass.iloc[0])
print('Training mass is equal across the current independent singleton source groups.')


## Inspect actual inner selection rather than choosing from anomaly results

The completed runner selects regularization from normal held-out source errors. Neural duration is chosen using the uncorrupted model and then shared across masking policies. Every inner source contributes equal weight even when validation folds have different sizes.

In [ ]:
result=completed_run();selection=read_json(artifact('selection.json'))
summary=[]
for record in selection.values():
    summary.append({'mode':record['mode'],'normal_training_sources':len(record['training_sources']),
                    'selected':record.get('selected_alpha',record.get('selected_alphas',record.get('selected_steps')))})
display(pd.DataFrame(summary).groupby('mode').agg(selection_records=('mode','size'),smallest_training_pool=('normal_training_sources','min'),largest_training_pool=('normal_training_sources','max')))
example=next(v for v in selection.values() if v['mode']=='ridge_posture_velocity')
display(example['inner_source_mse'])
assert not set(example['membership'][0]['train_sources'])&set(example['membership'][0]['validation_sources'])


## Check what was learned and what remains a separate hypothesis

The neural model has a coordinate-prediction objective in this study. Its hidden-feature variance and effective rank are measured on normal training prefixes to detect a collapsed or numerically unstable representation. An untrained-encoder coordinate probe is an additional reference. These checks do not by themselves establish clinically meaningful features.

In [ ]:
import joblib
ledger=read_json(artifact('model_fit_ledger.json'))
rows=[]
for item in ledger.values():
    identity=item['identity']
    if identity.get('kind')!='compact' or identity['config']['mode']!='mlp': continue
    if not any('outer0:mask_' in role for role in item['roles']): continue
    checkpoint=joblib.load(artifact(item['checkpoint']))
    rows.append({'policy':checkpoint['config']['mask_policy'],'seed':checkpoint['config']['seed'],
                 'parameters':checkpoint['parameter_count'],'steps':checkpoint['config']['steps'],
                 'effective_rank':checkpoint['diagnostics']['effective_rank'],
                 'mean_feature_std':checkpoint['diagnostics']['mean_feature_std']})
display(pd.DataFrame(rows))
print('JEPA/teacher, phase, learned-mask, and stochastic extensions require their own gate decision and run identity.')


## Keep inference independent of the withheld future

The public inference API receives exactly sixteen prefix samples. It always emits the same eight-time/eight-joint coordinate queries. Future values and future support are used later to measure error; they cannot enter prediction through an accidentally supplied full window.

In [ ]:
prediction=predict_forecaster(teaching_model,x[training[:2],:16],valid[training[:2],:16],actual_times[training[:2],:16])
assert prediction.shape==(2,8,8,2)
try:
    predict_forecaster(teaching_model,x[training[:2]],valid[training[:2]],actual_times[training[:2]])
except ValueError as exc:
    print('Full-window input was correctly rejected:',str(exc))
else:
    raise AssertionError('The predictor accepted future measurements.')

# Retain the exact inputs used by this independent-kernel execution.
input_path=PROJECT/'artifacts/executed-reference-v2/inputs/23_compact_models_and_fitting.json'
input_path.parent.mkdir(parents=True,exist_ok=True)
input_path.write_text(json.dumps({'path_base':'repository root','inputs':USED_INPUTS},indent=2)+'\n')
